In [1]:
import os, pickle

In [2]:
with open("../../../training_data/7.Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pickle.load(f)

len(extras_featuresd), extras_featuresd

(30,
 {'21du':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       21du               5             E          264            R   
  1       21du               5             E          265            R   
  2       21du               5             E          266            R   
  3       21du               5             E          267            R   
  4       21du               5             E          268            R   
  ..       ...             ...           ...          ...          ...   
  273     21du               5             E          546            R   
  274     21du               5             E          547            R   
  275     21du               5             E          548            R   
  276     21du               5             E          549            R   
  277     21du               5             E          550            R   
  
                      

# Make predictions

## Gather data/features

In [3]:
if not os.path.isfile("mkdssp-4.4.0-linux-x64"):
    os.system(f"ln -s {os.getcwd().rsplit('/', 3)[0]}/training_data/utils/external/mkdssp-4.4.0-linux-x64 mkdssp-4.4.0-linux-x64") # os.getcwd().rsplit('/', 3)[0] --> absolute path of allodb_new

In [4]:
import subprocess, pymol2

In [5]:
for pdb in extras_featuresd:
    # if pdb == "8aq6": continue
    pdb = pdb.upper()
    os.makedirs(pdb, exist_ok=True)

    # os.system(f"ln -s {os.getcwd().rsplit('/', 1)[0]}/structures/{pdb.lower()}.pdb {pdb}/{pdb}.pdb") # pdb file needs a HEADER first line for mkdssp to work
    pdbf = f"{pdb}/{pdb}.pdb"
    if not os.path.isfile(pdbf):
        with (
            open(f"{os.getcwd().rsplit('/', 1)[0]}/structures/{pdb.lower()}.pdb", "r") as orig_pdbf,
            open(pdbf, "w") as f
        ):
            f.write(f"HEADER {pdb}\n")
            f.write(orig_pdbf.read())
    
    # os.system(f"./mkdssp-4.4.0-linux-x64 --output-format dssp {pdb}/{pdb}.pdb") # need to capture output
    dsspf = pdbf.replace(".pdb", ".dssp")
    if not os.path.isfile(dsspf):
        with open(dsspf, "w") as f:
            f.write(
                subprocess.run(["./mkdssp-4.4.0-linux-x64", "--output-format=dssp", pdbf], capture_output=True)
                .stdout.decode()
            )

    asnf = pdbf.replace(".pdb", ".asn")
    if not os.path.isfile(asnf):
        for f in os.listdir(pdb):
            if f.endswith("-PSSM_Scoremat.asn"):
                os.system(f"ln -s {f} {asnf}")
                
    if not os.path.isfile(asnf):
        print(asnf)
        with pymol2.PyMOL() as pymol, open(pdbf.replace(".pdb", ".fasta"), "w") as f:#tempfile.NamedTemporaryFile("w+", suffix=".fasta") as f:
            pymol.cmd.load(pdbf, "prot")
            f.write(pymol.cmd.get_fastastr('prot'))
        
        # os.system(f"cd nr && psiblast -query {os.getcwd()}/{pdbf.replace('.pdb', '.fasta')} -db nr -out {os.getcwd()}/{pdbf.replace('.pdb', '.out')} -num_iterations 3 -evalue 0.001 -outfmt 11 -out_pssm {os.getcwd()}/{asnf} -save_pssm_after_last_round -num_threads 5")

## Predict

Edited utils.py:
- pocket filenames to be 1-indexed instead of 0-indexed `in_file = open(self.path + self.pdbid + '_out/pockets/pocket' + str(pockidx+1) + '_atm.pdb')`
- np.mean of neighboring PSSM etc... adjusted so that np.mean() uses axis=0 and does column-average
- function sequenceCode because in the .asn from local psiblast the sequence is hexadecimal-encoded and it needs to be decoded for the rest to work

Edited AllosESmain.py to adjust the relative location of `models.m` 

In [6]:
for pdb, feats in extras_featuresd.items():
    pdb = pdb.upper()

    asnf = f"{pdb}/{pdb}.asn"
    
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    if os.path.isfile(asnf) and not os.path.isfile(f"{pdb}/{pdb}_{chain}_result.csv"):
        os.system(f"cd {pdb} && python ../AllosES/AllosES/AllosESmain.py --PDBID {pdb} --CHAIN {chain}")

***** POCKET HUNTING BEGINS ***** 


***** POCKET HUNTING ENDS ***** 


******Detection pockets...******
Read successfully, PSSM matrix being extracted...
Extraction successful, exporting file now...
********************************************
******Program starts running...******
******Start extracting features...******
******Feature integration completed!******
******Start prediction...******
******Prediction completed!******
→The Pocket Rankings:
  Prediction Pocket1:	946:R,1028:R,949:R,945:R,1032:R,950:R,977:R,981:R,953:R,1031:R,1035:R,1027:R
  Prediction Pocket2:	956:R,1035:R,1039:R,953:R,1038:R,1031:R,1034:R
  Prediction Pocket3:	981:R,984:R,980:R,977:R,976:R
  Prediction Pocket4:	1076:R,1114:R,897:R,1118:R,898:R,895:R,951:R,901:R,886:R
  Prediction Pocket5:	1071:R,1113:R,1075:R,1109:R,1112:R,1102:R
  Prediction Pocket6:	988:R,985:R,984:R,1023:R,1024:R,1028:R,989:R
  Prediction Pocket7:	1099:R,1087:R,1084:R,852:R,855:R,856:R,853:R,1096:R,842:R,1093:R,1092:R,850:R,847:R,844:R,848:R,849:R
  Prediction Pocket8:	1125:R,1127:R,889:R,881:R,884:R,885:R,888

***** POCKET HUNTING BEGINS ***** 


***** POCKET HUNTING ENDS ***** 


...

***** POCKET HUNTING ENDS ***** 


******Detection pockets...******
Read successfully, PSSM matrix being extracted...
Extraction successful, exporting file now...
********************************************
******Program starts running...******
******Start extracting features...******
******Feature integration completed!******
******Start prediction...******
******Prediction completed!******
→The Pocket Rankings:
  Prediction Pocket1:	373:A,378:A,377:A,380:A,407:A,411:A,365:A,379:A,393:A,410:A,369:A,416:A,436:A,415:A,414:A
  Prediction Pocket2:	224:A,232:A,191:A,229:A,187:A,221:A,223:A,188:A,189:A,203:A,165:A,163:A,162:A,164:A,192:A,186:A,204:A,200:A,190:A,202:A,206:A
  Prediction Pocket3:	424:A,465:A,422:A,467:A,466:A,423:A,452:A,468:A,433:A,421:A,464:A,463:A,431:A
  Prediction Pocket4:	278:A,236:A,237:A,226:A,242:A,227:A,230:A,266:A,282:A,268:A,239:A
  Prediction Pocket5:	376:A,378:A,338:A,377:A,416:A,418:A,373:A,436:A,415:A
  Prediction Pocket6:	349:A,343:A,391:A,298:A,348:A,225:A,272:A,383:A,227:A,269:A,270:A,296:A

***** POCKET HUNTING BEGINS ***** 


***** POCKET HUNTING ENDS ***** 


******Detection pockets...******
Read successfully, PSSM matrix being extracted...
Extraction successful, exporting file now...
********************************************
******Program starts running...******
******Start extracting features...******
******Feature integration completed!******
******Start prediction...******
******Prediction completed!******
→The Pocket Rankings:
  Prediction Pocket1:	176:A,277:A,297:A,300:A,666:A,462:A,670:A,273:A,461:A,301:A,663:A,690:A,693:A,454:A
  Prediction Pocket2:	655:A,465:A,600:A,604:A,304:A,601:A,301:A,469:A,305:A,605:A,597:A,468:A,658:A,659:A,300:A,184:A,188:A,183:A,466:A,212:A,180:A,462:A,461:A,608:A
  Prediction Pocket3:	331:A,318:A,319:A,315:A,192:A,195:A,191:A,705:A,470:A,594:A,317:A,187:A,188:A,326:A,321:A,708:A,205:A,593:A,186:A,208:A,698:A,209:A,325:A,333:A,196:A,473:A,702:A,466:A,189:A,311:A,212:A,193:A,332:A,701:A,706:A,184:A,308:A,597:A,474:A,320:A
  Prediction Pocket4:	217:A,267:A,213:A,214:A,263:A,264:A,210:A
  Prediction Pocket

# Process

In [7]:
import pandas as pd

In [8]:
results = {}

for pdb, feats in extras_featuresd.items():
    pdb = pdb.upper()
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    csv = f"{pdb}/{pdb}_{chain}_result.csv"
    if os.path.isfile(csv):
        results[pdb.lower()] = {
            f"pocket{pocket.iloc[0]+1}": {
                "pro_ave": pocket["pro_ave"],
                "residues": pd.DataFrame(
                    (
                        res.split(":") 
                        for res in pocket["residues"].split(",")
                    ),
                    columns=["auth_seq_id", "auth_asym_id"]
                )
            }
            for i, pocket in pd.read_csv(csv).iterrows()
        }

len(results), results

(30,
 {'21du': {'pocket2': {'pro_ave': 0.2532559983232814,
    'residues':    auth_seq_id auth_asym_id
    0          946            R
    1         1028            R
    2          949            R
    3          945            R
    4         1032            R
    5          950            R
    6          977            R
    7          981            R
    8          953            R
    9         1031            R
    10        1035            R
    11        1027            R},
   'pocket7': {'pro_ave': 0.1434481120803093,
    'residues':   auth_seq_id auth_asym_id
    0         956            R
    1        1035            R
    2        1039            R
    3         953            R
    4        1038            R
    5        1031            R
    6        1034            R},
   'pocket6': {'pro_ave': 0.0929879126670484,
    'residues':   auth_seq_id auth_asym_id
    0         981            R
    1         984            R
    2         980            R
    3         977    

In [9]:
len(results), results

(30,
 {'21du': {'pocket2': {'pro_ave': 0.2532559983232814,
    'residues':    auth_seq_id auth_asym_id
    0          946            R
    1         1028            R
    2          949            R
    3          945            R
    4         1032            R
    5          950            R
    6          977            R
    7          981            R
    8          953            R
    9         1031            R
    10        1035            R
    11        1027            R},
   'pocket7': {'pro_ave': 0.1434481120803093,
    'residues':   auth_seq_id auth_asym_id
    0         956            R
    1        1035            R
    2        1039            R
    3         953            R
    4        1038            R
    5        1031            R
    6        1034            R},
   'pocket6': {'pro_ave': 0.0929879126670484,
    'residues':   auth_seq_id auth_asym_id
    0         981            R
    1         984            R
    2         980            R
    3         977    

In [10]:
resultsf = "alloses_results.pkl"

with open(resultsf, "wb") as f:
    pickle.dump(results, f)